# Bài 1 — Phân loại ransomware trên bộ dữ liệu BitcoinHeist (UCI)

**Nguồn:** UCI Machine Learning Repository — *BitcoinHeistRansomwareAddressDataset* (Akcora et al., 2019).
Mỗi dòng là một địa chỉ Bitcoin trong một ngày (2011–2018), mô tả bằng các đặc trưng đồ thị giao dịch;
nhãn `label` cho biết địa chỉ thuộc họ ransomware nào (hoặc `white` = không phải ransomware).

**Quy trình:**
1. Đọc & khám phá dữ liệu
2. Định nghĩa lại nhãn (gộp các lớp quá hiếm)
3. Feature engineering + chuẩn hoá (log1p → StandardScaler cho số, One-Hot cho phân loại)
4. **Baseline:** 4 mô hình (Logistic Regression, KNN, Random Forest, HistGradientBoosting)
5. **Cải thiện:** tỉ lệ undersampling, đặc trưng mới, tinh chỉnh siêu tham số, hiệu chỉnh trọng số lớp,
   XGBoost (GPU) / LightGBM, ensemble
6. Phân tích thêm: đánh giá nhị phân, vai trò của `year`, rò rỉ dữ liệu theo `address`

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.base import clone
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid")

DATA_PATH = "BitcoinHeistData.csv"
OUT_DIR = "outputs"
RANDOM_STATE = 42
TEST_SIZE = 0.2
N_JOBS = -1
os.makedirs(OUT_DIR, exist_ok=True)

## 1. Đọc dữ liệu & khám phá

In [ ]:
t0 = time.perf_counter()
df = pd.read_csv(DATA_PATH)
print(f"Đọc xong trong {time.perf_counter() - t0:.1f}s — shape = {df.shape}")
df.head()

In [ ]:
df.info()
print("\nGiá trị thiếu:", df.isna().sum().sum())
print("Số địa chỉ khác nhau:", df["address"].nunique())
df.describe().T

In [ ]:
label_counts = df["label"].value_counts()
print(f"Số lớp: {label_counts.size}")
display(label_counts.to_frame("count").assign(pct=lambda d: 100 * d["count"] / d["count"].sum()))

fig, ax = plt.subplots(figsize=(10, 7))
label_counts.sort_values().plot.barh(ax=ax, logx=True)
ax.set(title="Phân phối nhãn gốc (thang log)", xlabel="Số dòng (log)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/label_distribution.png", dpi=120); plt.show()

In [ ]:
# Số dòng ransomware theo năm → 'year' là đặc trưng rất mạnh.
# Lưu ý: số dòng 'white' mỗi năm đúng 365.000 (= 1.000/ngày) → 'white' được lấy mẫu cố định theo ngày.
ct = pd.crosstab(df["year"], df["label"] != "white").rename(columns={False: "white", True: "ransomware"})
ct["ransom_pct"] = 100 * ct["ransomware"] / ct.sum(axis=1)
ct

In [ ]:
NUM_BASE = ["length", "weight", "count", "looped", "neighbors", "income"]
fig, axes = plt.subplots(2, len(NUM_BASE), figsize=(20, 6))
sample = df[NUM_BASE].sample(200_000, random_state=RANDOM_STATE)
for i, c in enumerate(NUM_BASE):
    axes[0, i].hist(sample[c], bins=50); axes[0, i].set_title(f"{c} (gốc)")
    axes[1, i].hist(np.log1p(sample[c]), bins=50, color="tab:orange"); axes[1, i].set_title(f"log1p({c})")
plt.suptitle("Các đặc trưng số rất lệch → dùng log1p trước khi chuẩn hoá")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/numeric_distributions.png", dpi=120); plt.show()

## 2. Định nghĩa lại nhãn

Dữ liệu có 29 lớp, `white` chiếm ~98,6%, và ~20 lớp chỉ có < 100 mẫu (nhiều lớp 1–10 mẫu) — precision/recall
của những lớp đó vô nghĩa. Ta gộp thành **7 lớp**: `white`, 5 họ ransomware lớn nhất, và `otherRansom`.

In [ ]:
TOP_FAMILIES = label_counts.drop("white").head(5).index.tolist()
df["target"] = np.where(df["label"] == "white", "white",
                np.where(df["label"].isin(TOP_FAMILIES), df["label"], "otherRansom"))
CLASSES = sorted(df["target"].unique())
WHITE = "white"
df["target"].value_counts().to_frame("count")

## 3. Feature engineering

- Bỏ `address` (định danh, không phải đặc trưng).
- Ghép `year` + `day` → ngày cụ thể → tạo `month`, `weekday` (đặc trưng **categorical**).
- Đặc trưng **numerical** gốc: `length, weight, count, looped, neighbors, income`.
- **Đặc trưng mới** (dùng ở phần cải thiện):
  - `income_tz`: số chữ số 0 ở cuối của `income` (đơn vị satoshi) — đo độ "tròn" của số tiền nhận; tiền chuộc
    thường là số tròn.
  - `income_per_count`, `income_per_neighbor`, `weight_per_count`, `looped_ratio`: các tỉ lệ giữa đặc trưng gốc.

In [ ]:
date = pd.to_datetime(df["year"].astype(str) + "-01-01") + pd.to_timedelta(df["day"] - 1, unit="D")
df["month"] = date.dt.month
df["weekday"] = date.dt.dayofweek

inc_int = df["income"].round().astype("int64")
df["income_tz"] = sum((inc_int % 10**k == 0).astype("int8") for k in range(1, 13))
df["income_per_count"] = df["income"] / df["count"]
df["income_per_neighbor"] = df["income"] / df["neighbors"].clip(lower=1)
df["weight_per_count"] = df["weight"] / df["count"]
df["looped_ratio"] = df["looped"] / df["count"]

NUM_FE = NUM_BASE + ["income_tz", "income_per_count", "income_per_neighbor", "weight_per_count", "looped_ratio"]
CAT = ["year", "month", "weekday"]
X = df[NUM_FE + CAT]
y = df["target"]

# Độ "tròn" của income khác nhau rõ giữa white và ransomware
display(pd.crosstab(df["income_tz"].clip(upper=9), df["target"] == WHITE, normalize="columns")
          .rename(columns={True: "white", False: "ransomware"}).T.style.format("{:.3f}"))

## 4. Chia train / validation / test & cân bằng dữ liệu

- `train_test_split(stratify=y)` 80/20 → tập **test** giữ **phân phối thật**, chỉ dùng để báo cáo kết quả cuối.
- Tập train tiếp tục tách 80/20 → **train** / **validation**. Validation (phân phối thật) dùng để chọn tỉ lệ
  undersampling, siêu tham số và hiệu chỉnh trọng số lớp — không bao giờ chọn dựa trên tập test.
- Undersample lớp `white` **chỉ trên dữ liệu dùng để fit**.

In [ ]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=RANDOM_STATE)

def undersample(X_, y_, n_white, seed=RANDOM_STATE):
    # Giữ n_white mẫu 'white' (None = giữ tất cả) + toàn bộ ransomware
    if n_white is None:
        return X_, y_
    w = y_.index[y_ == WHITE]
    keep = np.concatenate([np.random.RandomState(seed).choice(w, n_white, replace=False), y_.index[y_ != WHITE]])
    return X_.loc[keep], y_.loc[keep]

print("train_full:", X_train_full.shape, "| train:", X_tr.shape, "| val:", X_val.shape, "| test:", X_test.shape)
pd.concat([y_train_full.value_counts().rename("train_full"), y_val.value_counts().rename("val"),
           y_test.value_counts().rename("test")], axis=1)

## 5. Tiền xử lý (chuẩn hoá)

`ColumnTransformer` nằm trong `Pipeline` nên scaler/encoder **chỉ fit trên tập train** (tránh rò rỉ dữ liệu).
One-Hot xuất ma trận **dense** (`sparse_output=False`) vì HistGradientBoosting không nhận dữ liệu sparse.

In [ ]:
def make_preprocessor(num_cols, cat_cols):
    return ColumnTransformer([
        ("num", Pipeline([("log1p", FunctionTransformer(np.log1p)),
                          ("scaler", StandardScaler())]), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

make_preprocessor(NUM_BASE, CAT)

## 6. Baseline: 4 mô hình

Cấu hình ban đầu: 6 đặc trưng số gốc + 3 đặc trưng phân loại, giữ 200.000 mẫu `white` khi train, không tinh chỉnh.

| Mô hình | Ý tưởng |
|---|---|
| Logistic Regression | Tuyến tính, softmax đa lớp; `class_weight="balanced"` |
| KNN | Dựa trên khoảng cách (k=5) — train gần như tức thì, predict chậm |
| Random Forest | Tập hợp nhiều cây quyết định (bagging) |
| HistGradientBoosting | Boosting trên cây với histogram — nhanh, mạnh với dữ liệu bảng |

In [ ]:
def log_progress(msg):
    line = f"{time.strftime('%H:%M:%S')} {msg}"
    print(line, flush=True)
    with open(f"{OUT_DIR}/progress.log", "a", encoding="utf-8") as f:
        f.write(line + "\n")

def summarize(name, y_true, y_pred, t_train, t_test):
    rep = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return rep, {"model": name, "accuracy": rep["accuracy"],
                 "weighted_precision": rep["weighted avg"]["precision"],
                 "weighted_recall": rep["weighted avg"]["recall"],
                 "weighted_f1": rep["weighted avg"]["f1-score"],
                 "macro_f1": rep["macro avg"]["f1-score"],
                 "binary_f1_ransom": f1_score(np.asarray(y_true) != WHITE, np.asarray(y_pred) != WHITE),
                 "train_time_s": t_train, "test_time_s": t_test}

def fit_pipeline(name, clf, X_fit, y_fit, num_cols, cat_cols=CAT):
    pipe = Pipeline([("pre", make_preprocessor(num_cols, cat_cols)), ("clf", clf)])
    log_progress(f"[{name}] fit ...")
    t0 = time.perf_counter(); pipe.fit(X_fit[num_cols + cat_cols], y_fit); t_train = time.perf_counter() - t0
    log_progress(f"[{name}] fit xong {t_train:.1f}s")
    return pipe, t_train

def evaluate(name, clf, X_fit, y_fit, X_te, y_te, num_cols, cat_cols=CAT, verbose=True):
    pipe, t_train = fit_pipeline(name, clf, X_fit, y_fit, num_cols, cat_cols)
    t0 = time.perf_counter(); y_pred = pipe.predict(X_te[num_cols + cat_cols]); t_test = time.perf_counter() - t0
    log_progress(f"[{name}] predict xong {t_test:.1f}s")
    rep, s = summarize(name, y_te, y_pred, t_train, t_test)
    if verbose:
        print(f"===== {name} =====  train {t_train:.1f}s | test {t_test:.1f}s")
        print(classification_report(y_te, y_pred, digits=4, zero_division=0))
    return pipe, y_pred, rep, s

BASELINE_MODELS = {
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "KNN": KNeighborsClassifier(n_neighbors=5, n_jobs=N_JOBS),
    "RandomForest": RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                                           n_jobs=N_JOBS, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1,
                                                           random_state=RANDOM_STATE),
}

X_base_fit, y_base_fit = undersample(X_train_full, y_train_full, 200_000)
base_preds, base_reports, base_rows = {}, {}, []
for name, clf in BASELINE_MODELS.items():
    _, base_preds[name], base_reports[name], s = evaluate(name, clf, X_base_fit, y_base_fit, X_test, y_test, NUM_BASE)
    base_rows.append(s)
baseline = pd.DataFrame(base_rows).set_index("model")
baseline.to_csv(f"{OUT_DIR}/summary_baseline.csv")
baseline

In [ ]:
# Mốc so sánh: luôn đoán 'white'
_, dummy = summarize("Luôn đoán white", y_test, np.full(len(y_test), WHITE), 0, 0)
pd.DataFrame([dummy]).set_index("model")[["accuracy", "weighted_f1", "macro_f1", "binary_f1_ransom"]]

**Nhận xét baseline:** accuracy/weighted F1 của mọi mô hình đều *thấp hơn* cách "luôn đoán white" (accuracy 0,986),
còn macro F1 chỉ ~0,13–0,47. Logistic Regression bị `class_weight="balanced"` + undersampling đẩy về phía
ransomware: recall cao nhưng precision chỉ vài %. → Cần cải thiện.

## 7. Cải thiện

### 7.1 Hiệu chỉnh trọng số lớp (class-weight / threshold tuning)

Undersampling làm mô hình "tưởng" ransomware phổ biến hơn thực tế. Thay vì `argmax(p)`, ta dự đoán
`argmax(p_k · w_k)` với vector trọng số `w` được tìm (coordinate ascent trên lưới log) để **tối đa macro F1 trên
tập validation** (phân phối thật). Trọng số này sau đó áp dụng nguyên vẹn lên tập test.

In [ ]:
CLS_IDX = {c: i for i, c in enumerate(CLASSES)}
K = len(CLASSES)

def macro_f1_fast(y_codes, p_codes):
    cm = np.bincount(y_codes * K + p_codes, minlength=K * K).reshape(K, K)
    tp = np.diag(cm); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    d = 2 * tp + fp + fn
    return float(np.mean(np.where(d > 0, 2 * tp / np.maximum(d, 1), 0)))

def tune_class_weights(P, y_true, passes=3):
    y_codes = np.asarray(pd.Series(y_true).map(CLS_IDX))
    w = np.ones(K); grid = np.exp(np.linspace(-5, 5, 41))
    best = macro_f1_fast(y_codes, P.argmax(1))
    for _ in range(passes):
        for k in range(K):
            for g in grid:
                w2 = w.copy(); w2[k] = g
                s = macro_f1_fast(y_codes, (P * w2).argmax(1))
                if s > best + 1e-9:
                    best, w = s, w2
    return w

def predict_weighted(pipe, X_, w, num_cols, cat_cols=CAT):
    # Cột của predict_proba theo thứ tự pipe.classes_ — với nhãn chuỗi hoặc mã số đều trùng thứ tự CLASSES
    P = pipe.predict_proba(X_[num_cols + cat_cols])
    return np.array(CLASSES)[(P * w).argmax(1)], P

def val_macro_f1(clf, n_white, num_cols, cat_cols=CAT, encode=False):
    # Fit trên train (undersample) → chỉnh w trên nửa val A → chấm macro F1 trên nửa val B
    Xf, yf = undersample(X_tr, y_tr, n_white)
    yf = yf.map(CLS_IDX) if encode else yf
    pipe, _ = fit_pipeline("val", clf, Xf, yf, num_cols, cat_cols)
    P = pipe.predict_proba(X_val[num_cols + cat_cols])
    half = np.arange(len(X_val)) % 2 == 0
    w = tune_class_weights(P[half], y_val[half])
    yv = np.asarray(y_val.map(CLS_IDX))
    return (macro_f1_fast(yv[~half], P[~half].argmax(1)),
            macro_f1_fast(yv[~half], (P[~half] * w).argmax(1)))

### 7.2 Chọn tỉ lệ undersampling

Thử giữ 100k / 200k / 500k / 1M / toàn bộ mẫu `white` khi train (HistGradientBoosting mặc định, đặc trưng gốc),
đo macro F1 trên validation trước và sau khi hiệu chỉnh trọng số lớp.

In [ ]:
rows = []
for nw in [100_000, 200_000, 500_000, 1_000_000, None]:
    if nw is not None and nw > (y_tr == WHITE).sum():
        continue
    raw, adj = val_macro_f1(HistGradientBoostingClassifier(max_iter=300, random_state=RANDOM_STATE), nw, NUM_BASE)
    rows.append({"n_white_train": nw or int((y_tr == WHITE).sum()), "val_macro_f1_raw": raw, "val_macro_f1_adjusted": adj})
us = pd.DataFrame(rows).set_index("n_white_train")
display(us)
ax = us.plot(marker="o", logx=True, figsize=(8, 4), title="Macro F1 (validation) theo số mẫu white khi train")
ax.set_xlabel("Số mẫu white trong tập train (log)")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/undersampling.png", dpi=120); plt.show()
N_WHITE = int(us["val_macro_f1_adjusted"].idxmax())
print("Chọn N_WHITE =", N_WHITE)

Giữ ít `white` cho kết quả tốt hơn: với quá nhiều `white`, loss bị lớp này chi phối và early stopping của
HistGradientBoosting dừng trước khi học được các lớp ransomware.

### 7.3 Đặc trưng mới

In [ ]:
feat_cmp = pd.DataFrame({
    "6 đặc trưng gốc": val_macro_f1(HistGradientBoostingClassifier(max_iter=300, random_state=RANDOM_STATE), N_WHITE, NUM_BASE),
    "11 đặc trưng (gốc + mới)": val_macro_f1(HistGradientBoostingClassifier(max_iter=300, random_state=RANDOM_STATE), N_WHITE, NUM_FE),
}, index=["val_macro_f1_raw", "val_macro_f1_adjusted"]).T
feat_cmp

### 7.4 Tinh chỉnh siêu tham số

Random search (script `experiments.py`, chạy trên server: 12 cấu hình HGB, 8 LightGBM, 10 XGBoost-GPU,
4 Random Forest), chọn theo macro F1 trên validation. Kết quả tìm kiếm được lưu ở `exp_out/results.json`;
cấu hình tốt nhất được dùng trực tiếp dưới đây.

In [ ]:
if os.path.exists("exp_out/results.json"):
    R = json.load(open("exp_out/results.json", encoding="utf-8"))
    search = pd.DataFrame([{"model": r["model"], "val_macro_f1": r["adjusted"]["macro_f1"],
                            "train_s": r["train_s"], **{f"p.{k}": v for k, v in r["params"].items()}}
                           for r in R if r["stage"] == "C_tuning"])
    display(search.sort_values(["model", "val_macro_f1"], ascending=[True, False])
                  .groupby("model").head(3).set_index("model").dropna(axis=1, how="all"))

In [ ]:
BEST_PARAMS = {
    "HistGradientBoosting": dict(learning_rate=0.03, max_leaf_nodes=255, min_samples_leaf=20,
                                 l2_regularization=10.0, max_features=0.8),
    "LightGBM": dict(learning_rate=0.05, num_leaves=31, min_child_samples=20, subsample=0.7,
                     subsample_freq=1, colsample_bytree=1.0, reg_lambda=1.0, n_estimators=600),
    "XGBoost-GPU": dict(n_estimators=300, learning_rate=0.05, max_depth=12, min_child_weight=1,
                        subsample=0.9, colsample_bytree=0.6, reg_lambda=5.0),
    "RandomForest": dict(n_estimators=300, min_samples_leaf=1, max_features="sqrt", max_depth=None),
}
IMPROVED_MODELS = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "KNN": KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=N_JOBS),
    "RandomForest": RandomForestClassifier(n_jobs=N_JOBS, random_state=RANDOM_STATE, **BEST_PARAMS["RandomForest"]),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE, early_stopping=True,
                                                           validation_fraction=0.1, n_iter_no_change=30,
                                                           max_iter=2000, **BEST_PARAMS["HistGradientBoosting"]),
    "LightGBM": LGBMClassifier(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1, **BEST_PARAMS["LightGBM"]),
    "XGBoost-GPU": XGBClassifier(device="cuda", tree_method="hist", random_state=RANDOM_STATE,
                                 eval_metric="mlogloss", **BEST_PARAMS["XGBoost-GPU"]),
}

### 7.5 Đánh giá cuối trên tập test

Mỗi mô hình: fit trên train (giữ `N_WHITE` mẫu white + đặc trưng mới) → chỉnh trọng số lớp trên validation →
dự đoán tập test. Thời gian test bao gồm cả bước nhân trọng số.

In [ ]:
X_fit, y_fit = undersample(X_tr, y_tr, N_WHITE)
pipes, weights, preds, probas_val, probas_test, reports, rows = {}, {}, {}, {}, {}, {}, []
for name, clf in IMPROVED_MODELS.items():
    y_fit_ = y_fit.map(CLS_IDX) if name == "XGBoost-GPU" else y_fit   # XGBoost cần nhãn dạng số
    pipe, t_train = fit_pipeline(name, clf, X_fit, y_fit_, NUM_FE)
    probas_val[name] = pipe.predict_proba(X_val[NUM_FE + CAT])
    weights[name] = tune_class_weights(probas_val[name], y_val)
    t0 = time.perf_counter()
    preds[name], probas_test[name] = predict_weighted(pipe, X_test, weights[name], NUM_FE)
    t_test = time.perf_counter() - t0
    log_progress(f"[{name}] predict xong {t_test:.1f}s")
    pipes[name] = pipe
    reports[name], s = summarize(name, y_test, preds[name], t_train, t_test)
    rows.append(s)
    print(f"===== {name} =====  train {t_train:.1f}s | test {t_test:.1f}s")
    print(classification_report(y_test, preds[name], digits=4, zero_division=0))

### 7.6 Ensemble

Trung bình xác suất của 3 mô hình boosting (HGB, LightGBM, XGBoost), rồi chỉnh trọng số lớp như trên.

In [ ]:
ENS = ["HistGradientBoosting", "LightGBM", "XGBoost-GPU"]
P_val = np.mean([probas_val[m] for m in ENS], axis=0)
P_test = np.mean([probas_test[m] for m in ENS], axis=0)
weights["Ensemble"] = tune_class_weights(P_val, y_val)
preds["Ensemble"] = np.array(CLASSES)[(P_test * weights["Ensemble"]).argmax(1)]
t_ens_train = sum(r["train_time_s"] for r in rows if r["model"] in ENS)
t_ens_test = sum(r["test_time_s"] for r in rows if r["model"] in ENS)
reports["Ensemble"], s = summarize("Ensemble", y_test, preds["Ensemble"], t_ens_train, t_ens_test)
rows.append(s)
print(classification_report(y_test, preds["Ensemble"], digits=4, zero_division=0))

summary = pd.DataFrame(rows).set_index("model")
summary.to_csv(f"{OUT_DIR}/summary_improved.csv")
summary

## 8. So sánh kết quả

In [ ]:
cmp = pd.concat({"Baseline": baseline[["accuracy", "weighted_f1", "macro_f1", "binary_f1_ransom"]],
                 "Cải thiện": summary[["accuracy", "weighted_f1", "macro_f1", "binary_f1_ransom"]]})
display(cmp.style.background_gradient(cmap="Greens", subset=["macro_f1", "binary_f1_ransom"]).format("{:.4f}"))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
m = pd.DataFrame({"Baseline": baseline["macro_f1"], "Cải thiện": summary["macro_f1"]})
m.plot.bar(ax=axes[0]); axes[0].set(title="Macro F1 trên tập test: baseline vs cải thiện", xlabel="")
axes[0].axhline(dummy["macro_f1"], ls="--", c="gray", label="Luôn đoán white"); axes[0].legend()
axes[0].tick_params(axis="x", rotation=20)
summary.drop(index="Ensemble")[["train_time_s", "test_time_s"]].plot.bar(ax=axes[1], logy=True)
axes[1].set(title="Thời gian train / test (giây, thang log) — mô hình cải thiện", xlabel="")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/model_comparison.png", dpi=120); plt.show()

In [ ]:
# Precision / Recall / F1 từng lớp — mô hình cải thiện
per_class = pd.concat({
    m: pd.DataFrame({c: reports[m][c] for c in CLASSES}).T[["precision", "recall", "f1-score", "support"]]
    for m in reports}, names=["model", "class"])
per_class.to_csv(f"{OUT_DIR}/per_class_improved.csv")
f1_pivot = per_class["f1-score"].unstack("model")[list(reports)]
display(f1_pivot.style.background_gradient(cmap="Greens", axis=None).format("{:.3f}"))

fig, ax = plt.subplots(figsize=(14, 5))
f1_pivot.plot.bar(ax=ax)
ax.set(title="F1-score từng lớp theo mô hình (đã cải thiện)", ylabel="F1", xlabel="")
plt.xticks(rotation=30, ha="right"); plt.tight_layout()
plt.savefig(f"{OUT_DIR}/f1_per_class.png", dpi=120); plt.show()

In [ ]:
best_name = summary["macro_f1"].idxmax()
print("Tốt nhất theo macro F1:", best_name)
fig, axes = plt.subplots(1, 2, figsize=(20, 8))
ConfusionMatrixDisplay.from_predictions(y_test, base_preds["HistGradientBoosting"], labels=CLASSES, normalize="true",
                                        values_format=".2f", cmap="Blues", ax=axes[0], colorbar=False)
axes[0].set_title("Baseline HistGradientBoosting")
ConfusionMatrixDisplay.from_predictions(y_test, preds[best_name], labels=CLASSES, normalize="true",
                                        values_format=".2f", cmap="Blues", ax=axes[1], colorbar=False)
axes[1].set_title(f"{best_name} (cải thiện)")
for ax in axes: ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/confusion_matrices.png", dpi=120); plt.show()

## 9. Phân tích thêm

### 9.1 Đánh giá nhị phân: phát hiện ransomware (bất kể họ nào)

In [ ]:
yt_bin = np.where(y_test == WHITE, "white", "ransomware")
yp_bin = np.where(preds[best_name] == WHITE, "white", "ransomware")
print(f"{best_name} — white vs ransomware")
print(classification_report(yt_bin, yp_bin, digits=4))

Phát hiện "có phải ransomware không" dễ hơn phân biệt *họ* ransomware: phần lớn lỗi còn lại là nhầm giữa các họ
(xem ma trận nhầm lẫn) và báo nhầm địa chỉ `white`.

### 9.2 Vai trò của `year`

In [ ]:
best_single = summary.drop(index="Ensemble")["macro_f1"].idxmax()
clf_noyear = clone(IMPROVED_MODELS[best_single])
y_fit_ = y_fit.map(CLS_IDX) if best_single == "XGBoost-GPU" else y_fit
pipe_ny, t_ny = fit_pipeline(f"{best_single} (no year)", clf_noyear, X_fit, y_fit_, NUM_FE, ["month", "weekday"])
w_ny = tune_class_weights(pipe_ny.predict_proba(X_val[NUM_FE + ["month", "weekday"]]), y_val)
p_ny, _ = predict_weighted(pipe_ny, X_test, w_ny, NUM_FE, ["month", "weekday"])
_, s_ny = summarize(f"{best_single} (no year)", y_test, p_ny, t_ny, 0)
pd.DataFrame([summary.loc[best_single].rename(best_single), pd.Series(s_ny).drop("model").rename(s_ny["model"])]
            )[["accuracy", "weighted_f1", "macro_f1", "binary_f1_ransom"]]

### 9.3 Rò rỉ dữ liệu theo `address`

Cùng một địa chỉ có thể xuất hiện nhiều ngày. Với phép chia ngẫu nhiên, một địa chỉ ransomware có thể nằm ở cả
train và test → mô hình "nhận ra" hành vi của chính địa chỉ đó → kết quả lạc quan. Kiểm tra mức chồng lấn và so
sánh với phép chia theo nhóm địa chỉ (`GroupShuffleSplit`, script `experiments_group.py`).

In [ ]:
addr = df["address"]
in_train = addr.loc[X_test.index].isin(set(addr.loc[X_train_full.index]))
is_ransom = y_test != WHITE
pd.Series({"Tất cả dòng test": in_train.mean(), "Dòng test ransomware": in_train[is_ransom].mean(),
           "Dòng test white": in_train[~is_ransom].mean()}, name="tỉ lệ có address xuất hiện trong train").to_frame()

In [ ]:
if os.path.exists("exp_out/group_results.json"):
    G = pd.DataFrame(json.load(open("exp_out/group_results.json", encoding="utf-8")))
    display(G.set_index(["split", "model"])[["macro_f1", "weighted_f1", "binary_f1_ransom"]])
    display(pd.DataFrame({(r["split"], r["model"]): r["per_class_f1"] for r in G.to_dict("records")}).round(3))

## 10. Lưu mô hình & kết quả (dùng lại cho Bài 3 — Streamlit)

In [ ]:
save_name = best_single
pipe_to_save = pipes[save_name]
if save_name == "XGBoost-GPU":
    pipe_to_save.named_steps["clf"].set_params(device="cpu")   # để chạy được trên máy không có GPU
joblib.dump({"pipeline": pipe_to_save, "class_weights": weights[save_name], "classes": CLASSES,
             "num": NUM_FE, "cat": CAT, "top_families": TOP_FAMILIES, "label_encoded": save_name == "XGBoost-GPU"},
            f"{OUT_DIR}/best_model.joblib", compress=3)
print("Đã lưu", save_name, "→", f"{OUT_DIR}/best_model.joblib")
print(sorted(os.listdir(OUT_DIR)))

## 11. Nhận xét

**Kết quả trên tập test (583.340 dòng, phân phối thật):**

| | Macro F1 | Weighted F1 | Accuracy | F1 nhị phân (ransomware) |
|---|---|---|---|---|
| Luôn đoán `white` | 0,142 | 0,979 | 0,986 | 0,000 |
| Baseline tốt nhất (HistGradientBoosting) | 0,456 | 0,975 | 0,967 | 0,361 |
| **XGBoost-GPU (cải thiện)** | **0,575** | 0,985 | 0,985 | 0,507 |
| **Ensemble HGB + LightGBM + XGBoost** | **0,576** | 0,985 | 0,984 | 0,503 |

- Macro F1 tăng **0,456 → 0,576** (+26%), F1 phát hiện ransomware tăng **0,361 → 0,503**. F1 từng họ đều tăng:
  CryptXXX 0,67 → 0,88; Locky 0,57 → 0,73; Cerber 0,43 → 0,63; CryptoWall 0,29 → 0,41; CryptoLocker 0,22 → 0,29.
- **Đóng góp của từng bước** (macro F1 trên validation): hiệu chỉnh trọng số lớp + giữ 100k mẫu white
  0,444 → 0,517; thêm đặc trưng mới → 0,544; tinh chỉnh siêu tham số → ~0,56–0,57.
  Hiệu chỉnh trọng số lớp cũng cứu được các mô hình yếu: Logistic Regression 0,129 → 0,347, KNN 0,284 → 0,416.
- **Thời gian:** KNN train nhanh nhất (0,2s) nhưng test chậm (14s); XGBoost trên GPU vừa tốt nhất vừa dự đoán nhanh
  nhất trong nhóm boosting (0,7s). HGB sau tinh chỉnh (255 lá, learning rate 0,03) train lâu hơn hẳn baseline (37s).
- **Accuracy không phù hợp** cho bài toán mất cân bằng này: mô hình tốt nhất vẫn chỉ ngang cách "luôn đoán white"
  về accuracy, trong khi macro F1 cho thấy khác biệt rõ.
- **`year` rất quan trọng:** bỏ `year` thì macro F1 giảm 0,575 → 0,444 (tỉ lệ ransomware khác nhau nhiều theo năm).
- **Rò rỉ theo `address`:** 54,7% dòng ransomware trong test có địa chỉ xuất hiện trong train, nhưng khi chia theo
  nhóm địa chỉ macro F1 chỉ giảm ~0,01 (XGBoost 0,575 → 0,565) → kết quả đáng tin cậy.
- **Hạn chế:** ở mức nhị phân, precision ransomware ~46% và recall ~56% — vẫn còn nhiều báo nhầm. Các họ
  CryptoLocker / CryptoWall khó tách nhau, `otherRansom` (gộp nhiều họ hiếm) gần như không học được (F1 ~0,1).
  Bộ dữ liệu chỉ có 6 đặc trưng đồ thị nên đây là giới hạn của dữ liệu nhiều hơn của mô hình.